# Ingesta de géneros

Lee `genre.csv` (CSV) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/genres` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

## Ingestion del archivo genre.csv

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee `genre.csv` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load(f"{bronze_folder_path}/genre.csv")
) 
display(genre_df)

In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])
genre_df = (
    spark.read
        .format('csv')
        .options("header", True)
        .schema(genre_schema)
        .load(f"{bronze_folder_path}/genre.csv")
)
display(genre_df)
genre_df.printSchema()

### Transformación de columnas

Renombra `genreId` y `genreName` a snake_case y agrega `ingestion_date` y `env`.

In [0]:
genre_df_final = (
    add_ingestion_date(genre_df)
    .select(
        col("genreId").alias("genre_id"),
        col("genreName").alias("genre_name"),
        lit(v_env).alias("env")
    )
)
display(genre_df_final)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/genres` en modo `overwrite`.

In [0]:
(genre_df_final.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{silver_folder_path}/genres")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")